# PAMAP2 Visualisation

This notebook uses the same real `.dat` parser as the dissertation pipeline,
with the hand IMU channels collapsed into the common schema.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
for candidate in (CWD, *CWD.parents):
    if (candidate / "pipeline").exists() and (candidate / "analysis").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repo root from the current working directory.")

for extra_path in (REPO_ROOT, REPO_ROOT / "analysis"):
    extra_str = str(extra_path)
    if extra_str not in sys.path:
        sys.path.insert(0, extra_str)

import notebook_utils as nb_utils

nb_utils.configure_matplotlib()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [ ]:
from pipeline.ingest import load_pamap2

DATASET_PATH = REPO_ROOT / "data" / "raw" / "PAMAP2_Dataset"
LOAD_FULL_DATASET = False
MAX_FILES = None if LOAD_FULL_DATASET else 3
INCLUDE_OPTIONAL = False

if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

df = load_pamap2(DATASET_PATH, max_files=MAX_FILES, include_optional=INCLUDE_OPTIONAL)
df = nb_utils.add_vector_magnitudes(df)
df["activity_name"] = df["label_raw"].astype(str).str.split(":", n=1).str[1].fillna(df["label_raw"].astype(str))

print(f"Loaded {len(df):,} rows from {DATASET_PATH}")
display(df.head())


In [ ]:
profile_df = nb_utils.dataset_profile(df)
loader_notes_df = pd.DataFrame({"loader_note": df.attrs.get("loader_notes", [])})
activity_name_df = nb_utils.count_table(df, "activity_name", top_n=12)
label_mapped_df = nb_utils.count_table(df, "label_mapped", top_n=10)
subject_df = nb_utils.count_table(df, "subject_id", top_n=10)

display(profile_df)
display(loader_notes_df if not loader_notes_df.empty else pd.DataFrame({"loader_note": ["No loader notes recorded"]}))
display(activity_name_df)
display(label_mapped_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
nb_utils.plot_count_bars(activity_name_df, "activity_name", ax=axes[0], title="Raw activity names", color="#3a86ff")
nb_utils.plot_count_bars(label_mapped_df, "label_mapped", ax=axes[1], title="Mapped HAR labels", color="#8338ec")
nb_utils.plot_count_bars(subject_df, "subject_id", ax=axes[2], title="Rows per subject", color="#ff006e")
fig.tight_layout()


In [ ]:
_ = nb_utils.plot_signal_histograms(df, dataset_label="PAMAP2")


In [ ]:
duration_df = nb_utils.session_duration_table(df)
display(duration_df)

fig, ax = plt.subplots(figsize=(10, 4))
ordered = duration_df.sort_values("duration_s", ascending=False, kind="stable")
ax.bar(ordered["session_id"].astype(str), ordered["duration_s"], color="#2a9d8f", alpha=0.9)
ax.set_title("Session durations by parsed file")
ax.set_xlabel("session_id")
ax.set_ylabel("duration_s")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()


In [ ]:
example_seq = nb_utils.pick_representative_sequence(df, preferred_label="locomotion", min_rows=256)
display(example_seq[["subject_id", "session_id", "timestamp", "ax", "ay", "az", "gx", "gy", "gz", "label_mapped"]].head(12))
_ = nb_utils.plot_sequence_axes(example_seq, title="Representative PAMAP2 sequence")
